# Choosing the detector: four open-vocabulary detectors outside the gate

Code of **Section V-A** of the paper (the detector comparison and Fig. 4). Grounding DINO, MM-Grounding-DINO, LLMDet and OWLv2 are
asked the same questions on the 70 development scenes of initial state 0: find the bowl, find the reference, find the bowl from the
full phrase ("the black bowl on the cookie box"), and find an object that is not on the table. This is what led to the hybrid
detector of Study 2: only OWLv2 never boxed absent objects, but it misses the wooden cabinet and the stove.

* **Part A** saves the development scenes once (image, depth and analysis-only ground truth) to `results/v6/scenes/`. It needs the
  simulator.
* **Part B** runs the four detectors on the saved scenes and scores them. It needs only the saved scenes and a GPU, no simulator.

*The printed output below each cell is the output of our own run (paths shortened; long progress logs shortened).*

## Part A: save the development scenes (about 30 min, resumable)
Run the shared set-up first (about 3 min).

In [ ]:
# Shared set-up: result folders, models, simulator and the gate's code
%run -i setup/s0_paths.py
%run -i setup/s1_models_and_simulator.py
%run -i setup/s2_engine.py
%run -i setup/s3_calibrate_camera.py
%run -i setup/s4_engine_fixes.py
%run -i setup/s5_policy_helpers.py
%run -i setup/s6_study1_setup.py

In [ ]:
# %% CELL v6_save_scenes  — saves every development scene once (image + depth + ground truth). ~15 min, resumable.
V6_DIR = os.path.join(RESULTS_ROOT, "v6")
SCENE_DIR = os.path.join(V6_DIR, "scenes")
os.makedirs(SCENE_DIR, exist_ok=True)
V6_STATES = [0, 1, 2]                      # 10 tasks x 3 states x 7 conditions = 210 scenes

def _uv6(cam, p, dz=0.02):
    r = cam.world_to_f(np.asarray(p, float) + np.array([0.0, 0.0, dz]))
    return None if r is None else [round(float(r[0]), 1), round(float(r[1]), 1)]

def _fixture_region6(env, cam, word):
    """image region of a fixed object (cabinet / stove) that has no position in obs: box around its projected parts"""
    m, d = env.sim.model, env.sim.data
    pts = []
    for gi in range(m.ngeom):
        bname = m.body_id2name(int(m.geom_bodyid[gi])) or ""
        if word in bname:
            r = cam.world_to_f(np.asarray(d.geom_xpos[gi], float))
            if r is not None:
                pts.append([r[0], r[1]])
    if not pts:
        return None
    pts = np.clip(np.array(pts, float), 0, IMG_W)
    x1, y1 = pts.min(0); x2, y2 = pts.max(0)
    return [round(float(v), 1) for v in (x1, y1, x2, y2)] if (x2 - x1 > 4 and y2 - y1 > 4) else None

_n, _t0 = 0, time.time()
for task_id in range(NUM_TASKS):
    todo = [(s, k) for s in V6_STATES for k in KINDS
            if not os.path.exists(os.path.join(SCENE_DIR, f"t{task_id}_s{s}_{k}.json"))]
    if not todo:
        continue
    env, _, clean_instr = get_task_env_and_instruction(task_id)
    for s, kind in todo:
        instr = perturb_instruction(clean_instr, kind, s) if kind in ("absent_reference", "absent_target") else clean_instr
        obs = setup_episode(env, task_id, s, perturb=kind if kind in SCENE_PERTURBATIONS else None, instruction=instr)
        cam = get_camera_model(env, obs)
        tkey, gkey = _true_keys(env, obs)
        parsed, goal = parse_instruction(instr), parse_goal(instr)
        refs = _as_list(parsed.get("reference"))
        gt_refs = {}
        for rp in refs:
            rk = _ref_key(rp, obs)
            word = "cabinet" if "cabinet" in rp else ("stove" if "stove" in rp else None)
            gt_refs[rp] = {"pt": _uv6(cam, obs[rk]) if rk else None,
                           "region": _fixture_region6(env, cam, word) if (rk is None and word) else None}
        sid = f"t{task_id}_s{s}_{kind}"
        Image.fromarray(np.ascontiguousarray(obs["agentview_image"][::-1, ::-1])).save(os.path.join(SCENE_DIR, sid + ".png"))
        np.savez_compressed(os.path.join(SCENE_DIR, sid + ".npz"),
                            depth=depth_as_2d(get_real_depth_map(env, obs["agentview_depth"])).astype(np.float32),
                            K=cam.K, cam2world=cam.cam2world, flips=np.array([cam.flip_u, cam.flip_v]))
        meta = {"id": sid, "task_id": task_id, "state": s, "kind": kind, "valid": int(kind not in INVALID_KINDS_V4),
                "instruction": instr, "relation": parsed.get("relation"), "target": parsed.get("target"), "refs": refs,
                "goal": goal.get("object") if goal else None,
                "refer": re.sub(r"^pick up\s+", "", instr.lower().split(" and place")[0]).strip(),
                "gt": {"target": _uv6(cam, obs[tkey]) if tkey else None,
                       "other_bowls": [_uv6(cam, obs[k]) for k in _object_keys(obs) if "bowl" in k and k != tkey],
                       "goal": _uv6(cam, obs[gkey]) if gkey else None, "refs": gt_refs}}
        with open(os.path.join(SCENE_DIR, sid + ".json"), "w") as f:      # written last = "done" marker
            json.dump(meta, f)
        _n += 1
        print(f"[{_n}] {sid:28s} refs={ {k: ('pt' if v['pt'] else 'region' if v['region'] else '-') for k, v in gt_refs.items()} } "
              f"({(time.time() - _t0) / _n:.1f}s each)")
    env.close()
OBS_HOOK = None
print("SCENES SAVED:", len([f for f in os.listdir(SCENE_DIR) if f.endswith('.json')]), "->", SCENE_DIR)

[1] t0_s0_clean                  refs={'plate': 'pt', 'ramekin': 'pt'} (7.8s each)
[2] t0_s0_absent_reference       refs={'red mug': '-', 'ramekin': 'pt'} (6.6s each)
[3] t0_s0_absent_target          refs={'plate': 'pt', 'ramekin': 'pt'} (6.2s each)
[4] t0_s0_relation_violation     refs={'plate': 'pt', 'ramekin': 'pt'} (6.9s each)
... (203 more lines of progress) ...
[208] t9_s2_distractor             refs={'wooden cabinet': 'region'} (7.9s each)
[209] t9_s2_position_change        refs={'wooden cabinet': 'region'} (8.0s each)
[210] t9_s2_occlusion              refs={'wooden cabinet': 'region'} (8.0s each)
SCENES SAVED: 210 -> results/v6/scenes


## Part B: the four detectors on the saved scenes
Start here if the scenes are already saved (only the result folder is needed).

In [ ]:
%run -i setup/s0_paths.py

In [ ]:
# ===== CELL 1: setup + load saved scenes =====
import os, json, time, re, gc, glob
import numpy as np, pandas as pd, torch
from PIL import Image
FAST_STATES = [0]            # [0] = 70 scenes (fastest).  Use [0, 1, 2] for all 210 saved scenes.
V6_DIR = os.path.join(RESULTS_ROOT, "v6")
SCENE_DIR = os.path.join(V6_DIR, "scenes")
DET_DIR = os.path.join(V6_DIR, "detections")
os.makedirs(DET_DIR, exist_ok=True)
device = "cuda"
ABSENT = {"red mug", "banana", "blue teapot"}
TABLE = {"table center", "table centre"}
SCENES = [json.load(open(f)) for f in sorted(glob.glob(os.path.join(SCENE_DIR, "*.json")))]
SCENES = [s for s in SCENES if s["state"] in FAST_STATES]

def load_img(sc):
    return Image.open(os.path.join(SCENE_DIR, sc["id"] + ".png")).convert("RGB")

def queries(sc):
    """what each model is asked for one scene: (tag, phrase)"""
    q = []
    if sc["target"]:
        q.append(("target", sc["target"]))                 # "black bowl" (or the absent object)
    for rp in sc["refs"]:
        if rp not in TABLE:
            q.append(("ref", rp))                          # "ramekin", "cookie box", "stove", ...
    if sc["valid"]:
        q.append(("refer", sc["refer"]))                   # "the black bowl on the cookie box"
    return list(dict.fromkeys(q))

print(len(SCENES), "scenes |", sum(len(queries(s)) for s in SCENES), "questions per model")

70 scenes | 181 questions per model


### Run the detectors (a few minutes, resumable; answers saved in `results/v6/detections/`)

In [ ]:
# ===== CELL 2: run the detectors one at a time (resumable; answers saved in results/v6/detections/) =====
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
RUN = ["gdino", "mm_gdino", "llmdet", "owlv2"]             # remove a name to skip that model

def gd_family(mid, old_style=False):
    """Grounding DINO, MM-Grounding-DINO and LLMDet share one code path."""
    def load():
        proc = AutoProcessor.from_pretrained(mid)
        mdl = AutoModelForZeroShotObjectDetection.from_pretrained(mid).to(device).eval()
        def detect(img, phrase):
            p = phrase.lower().strip()
            if old_style:
                inp = proc(images=img, text=p + ".", return_tensors="pt").to(device)
            else:
                inp = proc(images=img, text=[[p]], return_tensors="pt").to(device)
            with torch.no_grad():
                out = mdl(**inp)
            kw = {"text_threshold": 0.15} if old_style else {}
            r = proc.post_process_grounded_object_detection(out, threshold=0.15, target_sizes=[(img.height, img.width)], **kw)[0]
            return [b.tolist() + [float(s)] for b, s in zip(r["boxes"], r["scores"])]
        return detect, mdl
    return load

def load_owlv2():
    from transformers import Owlv2Processor, Owlv2ForObjectDetection
    mid = "google/owlv2-base-patch16-ensemble"
    proc = Owlv2Processor.from_pretrained(mid)
    mdl = Owlv2ForObjectDetection.from_pretrained(mid).to(device).eval()
    def detect(img, phrase):
        inp = proc(text=[[phrase if phrase.startswith("the ") else "a photo of a " + phrase]], images=img,
                   return_tensors="pt", truncation=True, max_length=16).to(device)
        with torch.no_grad():
            out = mdl(**inp)
        ts = torch.tensor([[max(img.size), max(img.size)]], device=device)
        try:
            r = proc.post_process_grounded_object_detection(outputs=out, target_sizes=ts, threshold=0.05)[0]
        except Exception:
            r = proc.post_process_object_detection(outputs=out, target_sizes=ts, threshold=0.05)[0]
        return [b.tolist() + [float(s)] for b, s in zip(r["boxes"], r["scores"])]
    return detect, mdl

LOADERS = {"gdino": gd_family("IDEA-Research/grounding-dino-base", old_style=True),
           "mm_gdino": gd_family("openmmlab-community/mm_grounding_dino_base_all"),
           "llmdet": gd_family("iSEE-Laboratory/llmdet_base"),
           "owlv2": load_owlv2}

for name in RUN:
    path = os.path.join(DET_DIR, name + ".jsonl")
    done = set()
    if os.path.exists(path):
        done = {(d["id"], d["tag"], d["phrase"]) for d in map(json.loads, open(path))}
    todo = [(sc, tag, ph) for sc in SCENES for tag, ph in queries(sc) if (sc["id"], tag, ph) not in done]
    print(f"\n=== {name}: {len(todo)} questions to run (already saved: {len(done)})")
    if not todo:
        continue
    try:
        detect, mdl = LOADERS[name]()
    except Exception as e:
        print(f"!! {name} could not be loaded -> skipped:\n   {type(e).__name__}: {str(e)[:600]}")
        continue
    t_start = time.time()
    for i, (sc, tag, ph) in enumerate(todo):
        t0 = time.time()
        try:
            boxes = detect(load_img(sc), ph)
        except Exception as e:
            print(f"!! {name} failed on {sc['id']} '{ph}' -> stopped:\n   {type(e).__name__}: {str(e)[:600]}")
            break
        boxes = sorted(boxes, key=lambda b: -b[4])
        with open(path, "a") as f:
            f.write(json.dumps({"id": sc["id"], "tag": tag, "phrase": ph, "sec": round(time.time() - t0, 3),
                                "boxes": [[round(float(v), 2) for v in b] for b in boxes]}) + "\n")
        if (i + 1) % 50 == 0 or i + 1 == len(todo):
            print(f"  [{i+1}/{len(todo)}] ~{(time.time() - t_start) / (i + 1) * (len(todo) - i - 1) / 60:.1f} min left")
    del detect, mdl
    gc.collect(); torch.cuda.empty_cache()
print("\nALL MODELS DONE ->", DET_DIR)


=== gdino: 0 questions to run (already saved: 543)

=== mm_gdino: 181 questions to run (already saved: 0)


  [50/181] ~0.8 min left
  [100/181] ~0.5 min left
  [150/181] ~0.2 min left
  [181/181] ~0.0 min left

=== llmdet: 181 questions to run (already saved: 0)


  [50/181] ~0.8 min left
  [100/181] ~0.5 min left
  [150/181] ~0.2 min left
  [181/181] ~0.0 min left

=== owlv2: 172 questions to run (already saved: 9)


  [50/172] ~0.6 min left
  [100/172] ~0.3 min left
  [150/172] ~0.1 min left
  [172/172] ~0.0 min left

ALL MODELS DONE -> results/v6/detections


### Score every detector (seconds)

In [ ]:
# ===== CELL 3: score every model (seconds) =====
THR = {"gdino": 0.22, "mm_gdino": 0.30, "llmdet": 0.30, "owlv2": 0.10}   # score cut-off per model (edit to try others)
BY_ID = {s["id"]: s for s in SCENES}
VALID_KINDS = ["clean", "distractor", "position_change", "occlusion"]

def inside(b, pt, pad=3):
    return pt is not None and b[0] - pad <= pt[0] <= b[2] + pad and b[1] - pad <= pt[1] <= b[3] + pad

def hits_ref(b, gt):
    if gt.get("pt") is not None:
        return inside(b, gt["pt"], pad=12)
    r = gt.get("region")
    if r is None:
        return None
    cx, cy = (b[0] + b[2]) / 2, (b[1] + b[3]) / 2
    return inside(r, (cx, cy), pad=10) or inside(b, ((r[0] + r[2]) / 2, (r[1] + r[3]) / 2), pad=0)

rows, per_task = [], {}
for f in sorted(glob.glob(os.path.join(DET_DIR, "*.jsonl"))):
    name = os.path.basename(f)[:-6]
    thr = THR.get(name, 0.0)
    D = {(d["id"], d["tag"], d["phrase"]): d for d in map(json.loads, open(f))}
    bowl, both, ref, absent, refer, other, none_, secs = [], [], [], [], [], [], [], []
    pt_rows = []
    for (sid, tag, ph), d in D.items():
        sc = BY_ID.get(sid)
        if sc is None:
            continue
        gt = sc["gt"]
        boxes = [b for b in d["boxes"] if b[4] >= thr]
        secs.append(d["sec"])
        if ph in ABSENT:                                    # object is not in the scene: any box is a false detection
            absent.append(int(len(boxes) > 0))
        elif tag == "target":
            bowl.append(int(any(inside(b, gt["target"]) for b in boxes)))
            both.append(int(bowl[-1] and all(any(inside(b, p) for b in boxes) for p in gt["other_bowls"])))
        elif tag == "ref":
            h = hits_ref(boxes[0], gt["refs"][ph]) if boxes else False
            if gt["refs"][ph].get("pt") is None and gt["refs"][ph].get("region") is None:
                h = None
            if h is not None:
                ref.append(int(bool(h)))
        elif tag == "refer" and sc["kind"] in VALID_KINDS:
            if not boxes:
                r = "none"
            else:
                b = boxes[0]
                on_t = inside(b, gt["target"]); on_o = any(inside(b, p) for p in gt["other_bowls"])
                r = "right" if (on_t and not on_o) else ("other" if (on_o and not on_t) else "none")
            refer.append(int(r == "right")); other.append(int(r == "other")); none_.append(int(r == "none"))
            pt_rows.append((sc["task_id"], int(r == "right")))
    m = lambda x: round(100 * float(np.mean(x)), 1) if len(x) else float("nan")
    n_q = sum(1 for k in D if k[0] in BY_ID)
    rows.append({"model": name, "questions": n_q,
                 "target bowl found %": m(bowl), "both bowls found %": m(both), "reference found %": m(ref),
                 "RIGHT bowl from full phrase %": m(refer), "picked other bowl %": m(other), "no / unclear box %": m(none_),
                 "false box for absent object % (lower=better)": m(absent), "sec per question": round(float(np.mean(secs)), 2)})
    per_task[name] = pd.DataFrame(pt_rows, columns=["task", "right"]).groupby("task").right.mean().mul(100).round(0)

tab = pd.DataFrame(rows)
print("STAGE 1 - detector comparison on the development scenes")
print(tab.T.to_string(header=False))
print("\nRIGHT bowl from the full phrase, per task (%):")
print(pd.DataFrame(per_task).to_string())
tab.to_csv(os.path.join(V6_DIR, "detector_comparison.csv"), index=False)

STAGE 1 - detector comparison on the development scenes
model                                         gdino  llmdet  mm_gdino  owlv2
questions                                       181     181       181    181
target bowl found %                            63.3    90.0      53.3   98.3
both bowls found %                             50.0    80.0      43.3   96.7
reference found %                              50.8    57.4      55.7   54.1
RIGHT bowl from full phrase %                  32.5    50.0      50.0   37.5
picked other bowl %                            65.0    42.5      40.0   37.5
no / unclear box %                              2.5     7.5      10.0   25.0
false box for absent object % (lower=better)  100.0   100.0     100.0    0.0
sec per question                                0.5    0.35      0.35   0.28

RIGHT bowl from the full phrase, per task (%):
      gdino  llmdet  mm_gdino  owlv2
task                                
0       0.0   100.0     100.0   25.0
1     100.0   1

### The selection rule fixed before looking at the numbers

In [ ]:
# ===== CELL 4: suggested winner (rule fixed BEFORE looking at the numbers) =====
# score = (RIGHT bowl from full phrase %) - (false box for absent object %); only models under MAX_SEC count; tie -> faster.
MAX_SEC = 5.0
t = tab.copy()
t["score"] = t["RIGHT bowl from full phrase %"].fillna(0) - t["false box for absent object % (lower=better)"].fillna(100)
t["fast enough"] = t["sec per question"] <= MAX_SEC
t = t.sort_values(["fast enough", "score", "sec per question"], ascending=[False, False, True])
print(t[["model", "RIGHT bowl from full phrase %", "false box for absent object % (lower=better)", "sec per question",
         "score", "fast enough"]].to_string(index=False))
print(f"\nSUGGESTED WINNER: {t.iloc[0]['model']}   (score {t.iloc[0]['score']:.1f}; current detector gdino = "
      f"{t[t.model == 'gdino'].score.iloc[0]:.1f})" if (t.model == "gdino").any() else f"\nSUGGESTED WINNER: {t.iloc[0]['model']}")
print("This is a suggestion from 70 scenes only - check the tables before deciding.")

   model  RIGHT bowl from full phrase %  false box for absent object % (lower=better)  sec per question  score  fast enough
   owlv2                           37.5                                           0.0              0.28   37.5         True
  llmdet                           50.0                                         100.0              0.35  -50.0         True
mm_gdino                           50.0                                         100.0              0.35  -50.0         True
   gdino                           32.5                                         100.0              0.50  -67.5         True

SUGGESTED WINNER: owlv2   (score 37.5; current detector gdino = -67.5)


### Pictures: one row per scene, one column per detector

In [ ]:
# ===== CELL 5: SEE the pictures — one row per scene, one column per model =====
SHOW_KINDS = ["clean", "absent_target"]          # which conditions to show (any of the 7 names)
SHOW_TASKS = range(10)                           # which tasks (0-9)
SHOW_STATE = FAST_STATES[0]
import matplotlib.pyplot as plt, matplotlib.patches as mpatches

DETS = {os.path.basename(f)[:-6]: {(d["id"], d["tag"], d["phrase"]): d for d in map(json.loads, open(f))}
        for f in sorted(glob.glob(os.path.join(DET_DIR, "*.jsonl")))}
names = list(DETS)
show = [s for s in SCENES if s["kind"] in SHOW_KINDS and s["task_id"] in SHOW_TASKS and s["state"] == SHOW_STATE]
THR4 = THR

def _in(b, pt, pad=3):
    return pt is not None and b[0] - pad <= pt[0] <= b[2] + pad and b[1] - pad <= pt[1] <= b[3] + pad

fig, axs = plt.subplots(len(show), len(names), figsize=(4.0 * len(names), 4.4 * len(show)), squeeze=False)
for r, sc in enumerate(show):
    img, gt = load_img(sc), sc["gt"]
    for c, name in enumerate(names):
        ax = axs[r, c]; ax.imshow(img); ax.axis("off")
        verdict = ""
        for tag, ph in queries(sc):
            d = DETS[name].get((sc["id"], tag, ph))
            if d is None:
                continue
            boxes = [b for b in d["boxes"] if b[4] >= THR4.get(name, 0.0)]
            if ph in ABSENT:                                         # object is NOT in the scene
                for b in boxes:
                    ax.add_patch(mpatches.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, ec="magenta", lw=2))
                verdict += f"'{ph}': " + ("WRONGLY FOUND" if boxes else "correctly not found") + "\n"
            elif tag == "refer":                                     # the bowl chosen from the full phrase
                if boxes:
                    b = boxes[0]
                    ok = _in(b, gt["target"]) and not any(_in(b, p) for p in gt["other_bowls"])
                    ax.add_patch(mpatches.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False,
                                                    ec="lime" if ok else "red", lw=2.5))
                    ax.text(b[0], b[3] + 9, f"{b[4]:.2f}", color="lime" if ok else "red", fontsize=8, weight="bold")
                    verdict += "bowl: " + ("RIGHT" if ok else "WRONG") + "\n"
                else:
                    verdict += "bowl: no box\n"
            elif tag == "ref" and boxes:                             # the reference object
                b = boxes[0]
                ax.add_patch(mpatches.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, ec="cyan", lw=1.5, ls="--"))
        if gt["target"] and sc["kind"] != "absent_target":
            ax.plot(gt["target"][0], gt["target"][1], "*", color="yellow", ms=12, mec="black")
        ax.set_title(f"{name}\n{verdict.strip()}", fontsize=8)
    axs[r, 0].text(0, -0.02, f"task {sc['task_id']} [{sc['kind']}]: {sc['instruction']}", transform=axs[r, 0].transAxes,
                   fontsize=7.5, va="top", style="italic")
plt.subplots_adjust(hspace=0.45, wspace=0.03)
out = os.path.join(V6_DIR, "detector_pictures.png")
plt.savefig(out, dpi=110, bbox_inches="tight"); plt.show()
print("yellow star = the correct bowl | green box = model chose the right bowl | red box = wrong bowl | "
      "cyan dashed = reference | magenta = box for an object that is not there\nsaved ->", out)

yellow star = the correct bowl | green box = model chose the right bowl | red box = wrong bowl | cyan dashed = reference | magenta = box for an object that is not there
saved -> results/v6/detector_pictures.png


### Fair cut-off check and which reference objects fail

In [ ]:
# ===== CELL 6: fair cut-off check + which reference objects fail (seconds; uses saved answers only) =====
ALL = {os.path.basename(f)[:-6]: [d for d in map(json.loads, open(f)) if d["id"] in BY_ID]
       for f in sorted(glob.glob(os.path.join(DET_DIR, "*.jsonl")))}
CUTS = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50, 0.60]

def at_cut(dets, c):
    bowl, both, absent = [], [], []
    for d in dets:
        gt = BY_ID[d["id"]]["gt"]
        boxes = [b for b in d["boxes"] if b[4] >= c]
        if d["phrase"] in ABSENT:
            absent.append(int(len(boxes) > 0))
        elif d["tag"] == "target":
            f = any(inside(b, gt["target"]) for b in boxes)
            bowl.append(int(f)); both.append(int(f and all(any(inside(b, p) for b in boxes) for p in gt["other_bowls"])))
    return 100 * np.mean(bowl), 100 * np.mean(both), 100 * np.mean(absent)

print("A) For each cut-off: target bowl found % / both bowls found % / FALSE box for absent object %")
rows = []
for name, dets in ALL.items():
    for c in CUTS:
        a, b, f = at_cut(dets, c)
        rows.append({"model": name, "cut-off": c, "bowl found %": round(a, 1), "both bowls %": round(b, 1), "false box %": round(f, 1)})
sweep = pd.DataFrame(rows)
for name in ALL:
    print(f"\n{name}"); print(sweep[sweep.model == name].drop(columns="model").to_string(index=False))

print("\nB) Best each model can do while keeping false boxes at or below 20%:")
best = []
for name in ALL:
    s = sweep[(sweep.model == name) & (sweep["false box %"] <= 20)]
    best.append({"model": name, **(s.sort_values("both bowls %", ascending=False).iloc[0].drop("model").to_dict() if len(s)
                                   else {"cut-off": None, "bowl found %": None, "both bowls %": None, "false box %": "never <= 20"})})
print(pd.DataFrame(best).to_string(index=False))

print("\nC) Reference found %, per reference object (top box, cut-offs from Cell 3):")
ref_rows = []
for name, dets in ALL.items():
    for d in dets:
        if d["tag"] != "ref" or d["phrase"] in ABSENT:
            continue
        g = BY_ID[d["id"]]["gt"]["refs"][d["phrase"]]
        if g.get("pt") is None and g.get("region") is None:
            continue
        boxes = [b for b in d["boxes"] if b[4] >= THR.get(name, 0.0)]
        ref_rows.append({"model": name, "reference": d["phrase"], "ok": int(bool(boxes and hits_ref(boxes[0], g)))})
rt = pd.DataFrame(ref_rows)
print((rt.pivot_table(index="reference", columns="model", values="ok", aggfunc="mean") * 100).round(0).to_string())
print("\ncases per reference:"); print(rt[rt.model == rt.model.iloc[0]].reference.value_counts().to_string())
sweep.to_csv(os.path.join(V6_DIR, "detector_cutoff_sweep.csv"), index=False)

A) For each cut-off: target bowl found % / both bowls found % / FALSE box for absent object %

gdino
 cut-off  bowl found %  both bowls %  false box %
    0.05          76.7          66.7        100.0
    0.10          76.7          66.7        100.0
    0.15          76.7          66.7        100.0
    0.20          70.0          56.7        100.0
    0.25          56.7          38.3        100.0
    0.30          50.0          26.7        100.0
    0.35          38.3          10.0         85.0
    0.40          30.0           5.0         35.0
    0.50          15.0           0.0          0.0
    0.60          11.7           0.0          0.0

llmdet
 cut-off  bowl found %  both bowls %  false box %
    0.05          98.3          96.7        100.0
    0.10          98.3          96.7        100.0
    0.15          98.3          96.7        100.0
    0.20          95.0          93.3        100.0
    0.25          91.7          90.0        100.0
    0.30          90.0          80.0     